Friday: the last mile, and the pivot that lied

> "Monday's growth review deck needs three things I can open on my laptop without a login: the
> revenue tree by segment for both quarters, the top-fifty protect list with a lookup so I can
> find any member by id, and one number on the front page with its trend. Nothing that needs
> Python. If a director changes an assumption in the room, the sheet must recalculate in front
> of them."
>
> Meera's chief of staff.

This notebook is the bridge rather than the deliverable. It builds the exports, proves which one
is safe, and then Excel does the rest.

**MAP** two exports  ->  **DO** total both  ->  **SEE** the doubling  ->
**CHECK** the lookup  ->  **SUM** the operating rule.

In [1]:
import pathlib
import sys

import pandas as pd

root = next(p for p in pathlib.Path.cwd().resolve().parents if (p / "scripts" / "c2kit.py").exists())
sys.path.insert(0, str(root / "scripts"))
import c2kit as kit

data = pathlib.Path.cwd().parent / "data"
clean = pd.read_csv(data / "C2_W02_D05_customer_table_STUDENT.csv")
raw = pd.read_csv(data / "C2_W02_D05_raw_export_STUDENT.csv")
print(f"clean export: {len(clean):,} rows, {clean.shape[1]} columns")
print(f"raw export  : {len(raw):,} rows, {raw.shape[1]} columns")

clean export: 300 rows, 6 columns
raw export  : 1,450 rows, 8 columns


## MAP. Two files, and only one of them is a customer table

They arrive in the same folder and look equally usable. The difference is grain: what one row
means. A file browser cannot show you that and a pivot will not ask.

In [2]:
kit.matrix(["clean export", "raw export"],
           ["one row is", "safe to pivot on"],
           [["one customer", "yes"],
            ["one order-payment pair", "no"]],
           title="Grain is the whole difference")
kit.check("the raw export holds more rows than the clean one", len(raw) > len(clean))

## DO. Total them both, the way a pivot would

A pivot sums the column it is given across the rows it is given. Nothing about that is clever
and nothing about it is wrong.

In [3]:
clean_total = clean["revenue"].sum()
raw_total = raw["order_amount"].sum()
print("clean export revenue:", kit.rupees(clean_total))
print("raw export revenue  :", kit.rupees(raw_total))
print(f"\nratio: {raw_total / clean_total:.4f}")
kit.check("the raw export roughly doubles the figure", 1.9 < raw_total / clean_total < 2.1,
          f"{raw_total / clean_total:.4f}")

clean export revenue: Rs 19,83,78,260
raw export revenue  : Rs 39,40,95,490

ratio: 1.9866


### This is Tuesday, in a spreadsheet

The 450 orders that carry two payment rows each appear twice in the raw export, and they are
the large invoices, so a pivot over that file roughly doubles the total. The fan-out did not go
away; it moved into a file that makes it invisible.

In [4]:
per_order = raw.groupby("order_id").size()
kit.flow(["1,000 orders", "joined to payments", f"{len(raw):,} rows",
          f"{(per_order > 1).sum()} counted twice"], lit=[3],
         title="What the export actually contains")
kit.check("450 orders appear more than once in the export", (per_order > 1).sum() == 450,
          f"{(per_order > 1).sum()}")

## SEE. The grain check anybody can run in ten seconds

Rows divided by distinct keys. Above one and the file is not what its name suggests.

In [5]:
for name, frame, key in (("clean", clean, "customer_id"), ("raw", raw, "customer_id")):
    ratio = len(frame) / frame[key].nunique()
    verdict = "one row per customer" if ratio <= 1.05 else "not a customer table"
    print(f"{name:>6}: {len(frame):,} rows / {frame[key].nunique():,} customers "
          f"= {ratio:.2f}  ->  {verdict}")
kit.check("the clean export really is one row per customer",
          len(clean) == clean["customer_id"].nunique())

 clean: 300 rows / 300 customers = 1.00  ->  one row per customer
   raw: 1,450 rows / 301 customers = 4.82  ->  not a customer table


## CHECK. The member who is not there

The customer table is built from orders, so a member who placed none in the two quarters is
absent. That is correct, and it is exactly what a lookup has to survive.

In [6]:
MISSING = "C-0170"
present = MISSING in set(clean["customer_id"])
print(f"{MISSING} in the clean table: {present}")
kit.check("one member id is absent, so the lookup has something to fail on", not present)

# What an approximate match would hand back: the nearest id above it.
nearest = sorted(c for c in clean["customer_id"] if c > MISSING)[0]
row = clean[clean["customer_id"] == nearest].iloc[0]
print(f"\nthe next id up is {nearest}, a real member in {row['segment']} "
      f"with {kit.rupees(row['revenue'])} of revenue")
kit.decision_ladder(["#N/A, which is ugly and truthful",
                     "a not-found message you wrote",
                     "the neighbour's row, which is tidy and wrong"],
                    cut_at=2, title="What the fourth argument decides")

C-0170 in the clean table: False



the next id up is C-0171, a real member in Retail-Plus with Rs 14,140 of revenue


### Why the neighbour is worse than the error

`#N/A` is ugly and truthful. A neighbouring member's data is tidy and wrong, and it carries a
real name, a real spend and a real segment. Nobody questions a row that looks complete.

## SUM. What goes to Excel, and what never does

Excel is where analysis meets its audience. The pivot must be built on a table that cannot
invent a number, the lookup must fail visibly, and the front-page figure must carry its
denominator, its period and its comparison.

In [7]:
q2 = kit.sql("SELECT sum(amount) s, count(*) n FROM orders WHERE quarter='Q2'")[0]
q1 = kit.sql("SELECT sum(amount) s FROM orders WHERE quarter='Q1'")[0]
change = 100 * (float(q2["s"]) - float(q1["s"])) / float(q1["s"])
print(f"{kit.rupees(q2['s'])} in Q2 across {q2['n']} orders, "
      f"against {kit.rupees(q1['s'])} in Q1, a change of {change:.1f} percent")
kit.check("the front-page figure carries a denominator, a period and a comparison", True,
          "orders, quarter, prior quarter")

kit.matrix(["warehouse", "pandas", "Excel"],
           ["owns", "never"],
           [["numbers Finance acts on", "exploration"],
            ["the analyst's iteration", "the source of truth"],
            ["presentation and poking", "cleaning, joining, the truth"]],
           title="The operating rule, and the fourth line is that Excel must be rebuildable")
kit.check_summary()

Rs 9,84,00,000 in Q2 across 462 orders, against Rs 10,00,00,000 in Q1, a change of -1.6 percent
